# core

> HTMX v4 support for FastHTML

In [ ]:
#| export
#| default_exp core

In [ ]:
#| export
import json

from fastcore.basics import patch
from fastcore.utils import *
from fastcore.xml import *
from fastcore.meta import delegates

from fasthtml.common import *
from fasthtml.starlette import *
from fasthtml.core import *

from fasthtml.core import _wrap_ex, _list, _get_htmx, _fix_anno, _find_wsp, _wrap_ws, _params, _handle, _ws_endp
from fasthtml.fastapp import _get_tbl, _app_factory

In [ ]:
from fasthtml.jupyter import *

In [ ]:
#| export
@delegates(ft_hx)
def Partial(*args, **kwargs): return ft_hx("hx-partial")(*args, **kwargs)


# htmx4 hdrs and metaCharacter

In [ ]:
#| export
htmx4src   = Script(src="https://unpkg.com/htmx.org@4.0.0-alpha7/dist/htmx.js")

In [ ]:
#| export
# When htmx4=True, configures htmx v4 with metaCharacter="-"
def def_hdrs(htmx=True, htmx4=False, surreal=True):
    "Default headers for a FastHTML app"
    hdrs = []
    if surreal: hdrs = [surrsrc,scopesrc] + hdrs
    if htmx and htmx4: raise ValueError("Cannot enable both htmx and htmx4")
    if htmx: hdrs = [htmxsrc,fhjsscr] + hdrs
    if htmx4: 
        # metaCharacter="-" makes htmx4 use dashes instead of colons (Python-friendly)
        meta_cfg = Meta(name="htmx:config", content=json.dumps({"metaCharacter": "-"}))
        hdrs = [meta_cfg, htmx4src,fhjsscr] + hdrs 
    # TODO: Check if fhjsscr works with htmx4
    return [charset, viewport] + hdrs

# FastHTML and fastapp

In [ ]:
#| export
# Patch FastHTML.__init__ to add htmx4 support
# - Adds `htmx4` parameter to toggle htmx v4 headers
# - Add self.htmx4
# - Passes htmx4 to def_hdrs() which handles the header selection
# - Maps 'ws' and 'ws4' extensions to 'ws4' when htmx4=True
@patch
def __init__(self: FastHTML, debug=False, routes=None, middleware=None, title: str = "FastHTML page", exception_handlers=None,
                on_startup=None, on_shutdown=None, lifespan=None, hdrs=None, ftrs=None, exts=None,
                before=None, after=None, surreal=True, htmx=True, htmx4=False, default_hdrs=True, sess_cls=SessionMiddleware,
                secret_key=None, session_cookie='session_', max_age=365*24*3600, sess_path='/',
                same_site='lax', sess_https_only=False, sess_domain=None, key_fname='.sesskey',
                body_wrap=noop_body, htmlkw=None, nb_hdrs=False, canonical=True, **bodykw):
    middleware,before,after = map(_list, (middleware,before,after))
    self.title,self.canonical,self.session_cookie,self.key_fname = title,canonical,session_cookie,key_fname
    self.htmx4 = htmx4
    hdrs,ftrs,exts = map(listify, (hdrs,ftrs,exts))
    if htmx4 and exts:
        exts = ['ws4' if e in ('ws', 'ws4') else e for e in exts]
    exts = {k:htmx_exts[k] for k in exts}
    htmlkw = htmlkw or {}
    if default_hdrs: hdrs = def_hdrs(htmx, htmx4, surreal=surreal) + hdrs
    hdrs += [Script(src=ext) for ext in exts.values()]
    if IN_NOTEBOOK:
        hdrs.append(iframe_scr) # TODO: check iframe_scr if work with htmx4
        from IPython.display import display,HTML
        if nb_hdrs: display(HTML(to_xml(tuple(hdrs))))
        middleware.append(cors_allow)
    on_startup,on_shutdown = listify(on_startup) or None,listify(on_shutdown) or None
    self.lifespan,self.hdrs,self.ftrs = lifespan,hdrs,ftrs
    self.body_wrap,self.before,self.after,self.htmlkw,self.bodykw = body_wrap,before,after,htmlkw,bodykw
    self.secret_key = get_key(secret_key, key_fname)
    if sess_cls:
        sess = Middleware(sess_cls, secret_key=self.secret_key,session_cookie=session_cookie,
                            max_age=max_age, path=sess_path, same_site=same_site,
                            https_only=sess_https_only, domain=sess_domain)
        middleware.append(sess)
    exception_handlers = ifnone(exception_handlers, {})
    if 404 not in exception_handlers:
        def _not_found(req, exc): return  Response('404 Not Found', status_code=404)
        exception_handlers[404] = _not_found
    excs = {k:_wrap_ex(v, k, hdrs, ftrs, htmlkw, bodykw, body_wrap=body_wrap) for k,v in exception_handlers.items()}
    super(FastHTML, self).__init__(debug, routes, middleware=middleware, exception_handlers=excs, on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan)

In [ ]:
#| export
# Supports htmx4=True for htmx v4 compatibility
def fast_app(
        db_file:Optional[str]=None, # Database file name, if needed
        render:Optional[callable]=None, # Function used to render default database class
        hdrs:Optional[tuple]=None, # Additional FT elements to add to <HEAD>
        ftrs:Optional[tuple]=None, # Additional FT elements to add to end of <BODY>
        tbls:Optional[dict]=None, # Experimental mapping from DB table names to dict table definitions
        before:Optional[tuple]|Beforeware=None, # Functions to call prior to calling handler
        middleware:Optional[tuple]=None, # Standard Starlette middleware
        live:bool=False, # Enable live reloading
        debug:bool=False, # Passed to Starlette, indicating if debug tracebacks should be returned on errors
        title:str="FastHTML page", # Default page title
        routes:Optional[tuple]=None, # Passed to Starlette
        exception_handlers:Optional[dict]=None, # Passed to Starlette
        on_startup:Optional[callable]=None, # Passed to Starlette
        on_shutdown:Optional[callable]=None, # Passed to Starlette
        lifespan:Optional[callable]=None, # Passed to Starlette
        default_hdrs=True, # Include default FastHTML headers such as HTMX script?
        pico:Optional[bool]=None, # Include PicoCSS header?
        surreal:Optional[bool]=True, # Include surreal.js/scope headers?
        htmx:Optional[bool]=True, # Include HTMX header?
        htmx4:Optional[bool]=False, # Include HTMX4 header?
        exts:Optional[list|str]=None, # HTMX extension names to include
        canonical:bool=True, # Automatically include canonical link?
        secret_key:Optional[str]=None, # Signing key for sessions
        key_fname:str='.sesskey', # Session cookie signing key file name
        session_cookie:str='session_', # Session cookie name
        max_age:int=365*24*3600, # Session cookie expiry time
        sess_path:str='/', # Session cookie path
        same_site:str='lax', # Session cookie same site policy
        sess_https_only:bool=False, # Session cookie HTTPS only?
        sess_domain:Optional[str]=None, # Session cookie domain
        htmlkw:Optional[dict]=None, # Attrs to add to the HTML tag
        bodykw:Optional[dict]=None, # Attrs to add to the Body tag
        reload_attempts:Optional[int]=1, # Number of reload attempts when live reloading
        reload_interval:Optional[int]=1000, # Time between reload attempts in ms
        static_path:str=".",  # Where the static file route points to, defaults to root dir
        body_wrap:callable=noop_body, # FT wrapper for body contents
        nb_hdrs:bool=False, # If in notebook include headers inject headers in notebook DOM?
        **kwargs):
    "Create a FastHTML or FastHTMLWithLiveReload app."
    h = (picolink,) if pico or (pico is None and default_hdrs) else ()
    if hdrs: h += tuple(hdrs)

    app = _app_factory(hdrs=h, ftrs=ftrs, before=before, middleware=middleware, live=live, debug=debug, title=title, routes=routes, exception_handlers=exception_handlers,
                  on_startup=on_startup, on_shutdown=on_shutdown, lifespan=lifespan, default_hdrs=default_hdrs, secret_key=secret_key, canonical=canonical,
                  session_cookie=session_cookie, max_age=max_age, sess_path=sess_path, same_site=same_site, sess_https_only=sess_https_only,
                  sess_domain=sess_domain, key_fname=key_fname, exts=exts, surreal=surreal, htmx=htmx, htmx4=htmx4, htmlkw=htmlkw,
                  reload_attempts=reload_attempts, reload_interval=reload_interval, body_wrap=body_wrap, nb_hdrs=nb_hdrs, **(bodykw or {}))
    app.static_route_exts(static_path=static_path)
    if not db_file: return app,app.route

    db = database(db_file)
    if not tbls: tbls={}
    if kwargs:
        if isinstance(first(kwargs.values()), dict): tbls = kwargs
        else:
            kwargs['render'] = render
            tbls['items'] = kwargs
    dbtbls = [_get_tbl(db.t, k, v) for k,v in tbls.items()]
    if len(dbtbls)==1: dbtbls=dbtbls[0]
    return app,app.route,*dbtbls

# WS

## Migrating WebSockets from htmx v2 to v4

**Key Differences:**

**htmx v2:**
- Requires WS extension: `Script(src="https://cdn.jsdelivr.net/npm/htmx-ext-ws@2.0.3/ws.js")`
- Uses `hx_ext="ws"`, `ws_connect="/endpoint"`, `ws_send=True`
- Server sends raw HTML directly: `await send(Div('Hello', id='msg'))`
- Client receives HTML and swaps based on `id` or `hx-swap-oob`

**htmx v4:**
- Requires WS extension: `Script(src="https://unpkg.com/htmx.org@4.0.0-alpha7/dist/ext/hx-ws.js")`
- Uses `hx_ws_connect="/endpoint"`, `hx_ws_send=True`
- Server sends raw HTML, same as v2 — uses `hx_swap_oob=True` or `<hx-partial>` for targeting
- No need for `hx_ext="ws"` attribute — extension auto-registers when loaded
- **Must explicitly set `hx_swap_oob=True`** on each element for ID-based swapping. In v2, elements with a matching `id` would swap implicitly; in v4 you must be explicit.

**Receiving form data (server-side):**
- **v2:** Form fields are at the **root** of the incoming JSON — e.g. `data['message']`
- **v4:** Form fields are nested under `values` — e.g. `data['values']['message']`

This is because v4 wraps everything in a JSON envelope with `type`, `request_id`, `headers`, `values`, etc., even when the *server* sends raw HTML back.

**Important:** Although htmx v4 WS supports a JSON envelope format for *server→client* messages too, we send raw HTML for compatibility with htmx v2. This means the same `_send_ws` function works for both versions. The main differences are the attribute names and the incoming data structure described above.

## Summary

Patches needed to make WS work with both htmx v2 and v4:

## 1. **Headers & Configuration**
- Created `htmx4src` pointing to htmx v4 alpha
- Modified `def_hdrs()` to add htmx v4 script + meta config with `metaCharacter="-"` (makes attributes use dashes instead of colons)
- Added `ws4` extension URL to `htmx_exts`

## 2. **FastHTML.__init__ Patch**
- Added `htmx4` parameter and stored it as `self.htmx4`
- Auto-maps `'ws'` extension to `'ws4'` when `htmx4=True`

## 3. **WebSocket Parameter Handling** (`_find_wsp_patch`)
- Checks both top-level `data.get(arg)` (htmx v2) AND `data.get('values', {}).get(arg)` (htmx v4)
- Always uses `_send_ws` (raw HTML) for both v2 and v4 — no separate `_send_ws4` needed

In [ ]:
#| export
from inspect import Parameter
empty = Parameter.empty

In [ ]:
#| export
import fasthtml.core as _core
from fasthtml.core import _send_ws

In [ ]:
#| export
from typing import Optional, get_type_hints, get_args, get_origin, Union, Mapping, TypedDict, List, Any
from types import UnionType, SimpleNamespace as ns, GenericAlias
from datetime import datetime,date


In [ ]:
from starlette.testclient import TestClient

In [ ]:
htmx_exts

In [ ]:
#| export
htmx_exts['ws4'] = 'https://unpkg.com/htmx.org@4.0.0-alpha7/dist/ext/hx-ws.js'

## WS4 patch

Only `_find_wsp` needs patching — to check `data['values']` for htmx v4 form data. No separate send function needed since we use raw HTML.

In [ ]:
#| export
# Patch for htmx v4 WebSocket: form fields are now in data['values'] instead of top-level data
def _find_wsp_patch(ws, data, hdrs, arg:str, p:Parameter, htmx4=False):
    "In `data` find param named `arg` of type in `p` (`arg` is ignored for body types)"
    anno = p.annotation
    if isinstance(anno, type):
        if issubclass(anno, HtmxHeaders): return _get_htmx(hdrs)
        if issubclass(anno, Starlette): return ws.scope['app']
        if issubclass(anno, WebSocket): return ws
        if issubclass(anno, dict): return data
    if anno is empty:
        if arg.lower()=='ws': return ws
        if arg.lower()=='scope': return dict2obj(ws.scope)
        if arg.lower()=='data': return data
        if arg.lower()=='htmx': return _get_htmx(hdrs)
        if arg.lower()=='app': return ws.scope['app']
        if arg.lower()=='send': return partial(_send_ws, ws)
        if 'session'.startswith(arg.lower()): return ws.scope.get('session', {})
        return None
    res = data.get(arg, None)  # htmx v2: top-level
    if res is empty or res is None: res = data.get('values', {}).get(arg, None)  # htmx v4: in 'values'
    if res is empty or res is None: res = hdrs.get(arg, None)
    if res is empty or res is None: res = p.default
    if not isinstance(res, (list,str)) or anno is empty: return res
    return [_fix_anno(anno, o) for o in res] if isinstance(res,list) else _fix_anno(anno, res)

_core._find_wsp = _find_wsp_patch

## WS Example

### WS Example with htmx2

```python
from asyncio import sleep

app = FastHTML(exts='ws')
rt = app.route

def mk_inp(): return Input(id='msg')
nid = 'notifications'

@rt('/')
async def get():
    cts = Div(
        Div(id=nid),
        Form(mk_inp(), id='form', ws_send=True),
        hx_ext='ws', ws_connect='/ws')
    return Titled('Websocket Test', cts)

async def on_connect(send): await send(Div('Hello, you have connected', id=nid))
async def on_disconnect( ): print('Disconnected!')

@app.ws('/ws', conn=on_connect, disconn=on_disconnect)
async def ws(msg:str, send):
    await send(Div('Hello ' + msg, id=nid))
    await sleep(2)
    return Div('Goodbye ' + msg, id=nid), mk_inp()

srv = JupyUvi(app)
```

### WS Example with htmx v4

```python
from asyncio import sleep

app = FastHTML(exts='ws', htmx=False, htmx4=True)
rt = app.route

def mk_inp(): return Input(id='msg')
nid = 'notifications'

@rt('/')
async def get():
    cts = Div(
        Div(id=nid),
        Form(mk_inp(), id='form', hx_ws_send=True),
        hx_ws_connect='/ws')
    return Titled('Websocket Test', cts)

async def on_connect(send): 
    await send(Div('Hello, you have connected', id=nid, hx_swap_oob=True))
async def on_disconnect(): print('Disconnected!')

@app.ws('/ws', conn=on_connect, disconn=on_disconnect)
async def ws(msg:str, send):
    await send(Div('Hello ' + msg, id=nid, hx_swap_oob=True))
    await sleep(2)
    # Send multiple elements in one message
    return (
        Div('Goodbye ' + msg, id=nid, hx_swap_oob=True),
        Input(id='msg', name='msg', value='', hx_swap_oob=True)
    )

srv = JupyUvi(app)
```

# SSE

## Migrating SSE from htmx v2 to v4

**Key Differences:**

**htmx v2:**
- Requires SSE extension: `Script(src="https://unpkg.com/htmx-ext-sse@2.2.1/sse.js")`
- Uses `hx_ext="sse"`, `sse_connect="/endpoint"`, `sse_swap="message"`
- Server uses `sse_message(data)` which includes `event: message\n`

**htmx v4:**
- SSE built into core htmx.js, no separate extension file needed
- Uses regular htmx attributes: `hx_get="/endpoint"`, `hx_trigger="load"`
- Server must omit `event:` line for normal swaps: `f"data: {to_xml(data)}\n\n"`

**Important:** In htmx v4, including `event:` triggers a custom DOM event instead of swapping content. For basic SSE swapping, use only `data:` lines.

## SSE patch

In [ ]:
#| export
def sse_message(elm, event='message', htmx4=False):
    "Convert element `elm` into a format suitable for SSE streaming"
    data = '\n'.join(f'data: {o}' for o in to_xml(elm).splitlines())
    # In htmx v4, omitting event: does default swap; including it triggers custom DOM event
    if htmx4 and event == 'message': return f'{data}\n\n'
    return f'event: {event}\n{data}\n\n'

import fasthtml.components
fasthtml.components.sse_message = sse_message

## SSE Example

### SSE Example with htmx v2

```python
import random
from asyncio import sleep

hdrs=(Script(src="https://unpkg.com/htmx-ext-sse@2.2.1/sse.js"),)
app,rt = fast_app(hdrs=hdrs)

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate random numbers, as the list grows scroll downwards."),
        Div(hx_ext="sse",
            sse_connect="/number-stream",
            hx_swap="beforeend show:bottom",
            sse_swap="message"))

shutdown_event = signal_shutdown()

async def number_generator():
    while not shutdown_event.is_set():
        data = Article(random.randint(1, 100))
        yield sse_message(data)
        await sleep(1)

@rt("/number-stream")
async def get(): return EventStream(number_generator())

srv = JupyUvi(app)
```

### SSE Example with htmx v4

```python
import random
from asyncio import sleep

app,rt = fast_app(htmx=False, htmx4=True)

@rt
def index():
    return Titled("SSE Random Number Generator",
        P("Generate random numbers, as the list grows scroll downwards."),
        Div(hx_get="/number-stream",
            hx_trigger="load",
            hx_swap="beforeend show:bottom"))

shutdown_event = signal_shutdown()

async def number_generator():
    while not shutdown_event.is_set():
        data = Article(random.randint(1, 100))
        yield sse_message(data, htmx4=True)
        await sleep(1)

@rt("/number-stream")
async def get(): return EventStream(number_generator())

srv = JupyUvi(app)
```

# Dev

Search for me the list of project we haven't migrate to htmx4 yet and give me one

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let me check what example folders exist and which ones already have a v4 migration dialog.

<details class='tool-usage-details'>
<summary>view(path='.')</summary>

```json
{
  "id": "toolu_01ULYF2XmPX7RqjYbgGsPrpg",
  "call": {
    "function": "view",
    "arguments": {
      "path": "."
    }
  },
  "result": "'Directory contents of /app/data/fasthtml-example/hx4_patch/nbs:\\n/app/data/fasthtml-example/hx4_patch/nbs/htmx4_minimal.ipynb (9.0k)\\n/app/data/fasthtml-example/hx4_patch/nbs/sse_basic.ipynb (32.5k)\\n/app/data/fasthtml-example/hx4_patch/nbs/00_core.ipynb (25.8k)\\n/app/data/fasthtml-example/hx4_patch/nbs/styles.css (0.6k)\\n/app/data/fasthtml-example/hx4_patch/nbs/index.ipynb (0.5k)\\n/app/data/fasthtml-example/hx4_patch/nbs/nbdev.yml (0.2k)'"
}
```

</details>



<details class='tool-usage-details'>
<summary>view(path='/app/data/fasthtml-example')</summary>

```json
{
  "id": "toolu_01RCbbBFpByKK7EggfT9vgZW",
  "call": {
    "function": "view",
    "arguments": {
      "path": "/app/data/fasthtml-example"
    }
  },
  "result": "'Directory contents of /app/data/fasthtml-example:\\n/app/data/fasthtml-example/02_chatbot \ud83d\udcc1\\n/app/data/fasthtml-example/todos2 \ud83d\udcc1\\n/app/data/fasthtml-example/devtools \ud83d\udcc1\\n/app/data/fasthtml-example/e_commerce \ud83d\udcc1\\n/app/data/fasthtml-example/todos2-hf \ud83d\udcc1\\n/app/data/fasthtml-example/chess_app \ud83d\udcc1\\n/app/data/fasthtml-example/LICENSE (11.1k)\\n/app/data/fasthtml-example/h2f \ud83d\udcc1\\n/app/data/fasthtml-example/image_app_simple \ud83d\udcc1\\n/app/data/fasthtml-example/code_editor \ud83d\udcc1\\n/app/data/fasthtml-example/cli_oauth_example \ud83d\udcc1\\n/app/data/fasthtml-example/CRAFT.ipynb (145.2k)\\n/app/data/fasthtml-example/README.md (2.2k)\\n/app/data/fasthtml-example/code_highlight_and_copy \ud83d\udcc1\\n/app/data/fasthtml-example/01_todo_app \ud83d\udcc1\\n/app/data/fasthtml-example/hacker_news_reskin \ud83d\udcc1\\n/app/data/fasthtml-example/file_upload_form_example \ud83d\udcc1\\n/app/data/fasthtml-example/htmx \ud83d\udcc1\\n/app/data/fasthtml-example/passkey \ud83d\udcc1\\n/app/data/fasthtml-example/shoelace.py (2.8k)\\n/app/data/fasthtml-example/00_game_of_life \ud83d\udcc1\\n/app/data/fasthtml-example/3d-compo<TRUNCATED>"
}
```

</details>



🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's what I found. The examples that **already have** an htmx v4 migration (file with `4` suffix):

| Folder | v4 file(s) |
|--------|-----------|
| `00_game_of_life` | `gol4.ipynb` |
| `01_todo_app` | `todo4.ipynb` |
| `02_chatbot` | `basic4.py`, `basic4.ipynb`, `ws4.py`, `polling4.py` |

Everything else is **not yet migrated** — that's about 25+ examples, including `03_pictionary`, `04_sse` (multiple files), `todos1`, `todos2`, `e_commerce`, `infinite_scroll`, `doodle`, `hacker_news_reskin`, `annotate_text`, `code_editor`, `data_spot_check`, `image_app_simple`, `chess_app`, `helloworld`, `echo`, `h2f`, `story`, `xtermjs`, and more.

A good simple one to start with might be **`helloworld`** or **`echo`** — they're tiny and would let you verify the basic htmx4 setup works. Or if you want something a bit more interesting, **`infinite_scroll`** or **`todos1`** would exercise more htmx features (swaps, triggers, targets).

Which one catches your eye?

Yeah, let's try echo. Give me the code that works with htmx2 here. I guess it should just works and we only need to serve use JupyUvi instead of serve

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->





🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>view(path='/app/data/fasthtml-example/echo')</summary>

```json
{
  "id": "toolu_01SVhzY4xjyavu9Hee5NAqHw",
  "call": {
    "function": "view",
    "arguments": {
      "path": "/app/data/fasthtml-example/echo"
    }
  },
  "result": "'Directory contents of /app/data/fasthtml-example/echo:\\n/app/data/fasthtml-example/echo/requirements.txt (0.0k)\\n/app/data/fasthtml-example/echo/main.py (0.2k)'"
}
```

</details>



<details class='tool-usage-details'>
<summary>view(path='/app/data/fasthtml-example/echo/main.py')</summary>

```json
{
  "id": "toolu_016eGeBMMPhPjuPpUyiSjWQM",
  "call": {
    "function": "view",
    "arguments": {
      "path": "/app/data/fasthtml-example/echo/main.py"
    }
  },
  "result": "'from fasthtml.common import *\\n\\napp = FastHTML(middleware=[cors_allow])\\nrt = app.route\\n\\n@rt\\ndef index(d:dict, q:str=\"\"):\\n    print(str(d)[:1000])\\n    return q\\n\\nserve()\\n'"
}
```

</details>

Nice and simple! Here it is adapted for solveit (just swapping `serve()` for `JupyUvi`):

```python
from fasthtml.common import *
from fasthtml.jupyter import *

app = FastHTML(middleware=[cors_allow])
rt = app.route

@rt
def index(d:dict, q:str=""):
    print(str(d)[:1000])
    return q

srv = JupyUvi(app)
```

This should work as-is with htmx v2 (the default). It's a bare-bones echo — it just returns whatever `q` query param you pass. Try running it and hitting the URL with `?q=hello` to confirm it works before we migrate it.

## HTMX 2

In [ ]:
from fasthtml.common import *
from fasthtml.jupyter import *

app = FastHTML(middleware=[cors_allow])
rt = app.route

@rt
def index(d:dict, q:str=""):
    print(str(d)[:1000])
    return q

srv = JupyUvi(app)
